# logistic regression

logistic regression example. logistic regression is an algorithim for binary classiffication

Gradient looks like General Update Rule: $$ w=w−α \cdot \frac{∂L}{∂w}$$

 
where $$α:learning rate \\ L:loss function\\ \frac{∂L}{∂w}:gradients of loss$$
 
 

## Adam optimizer

Algorithims that calculate the partial derivative of the weights and average them are known as SDG (Stochastic Gradient Descent) Optimiziers, whereas Adam (adaptive movement estimation) Optimizers change the learning rate based on past gradients.
$$ 
w_{j,t+1}=w_{j,t}-\alpha \cdot \frac{\hat{m}_{t,w_j}}{\sqrt{\hat{v}_{t,w_j}}+\epsilon}
$$ 
where

$\hat{m}_t$ : moment at iteration t with bias correction

$\beta_1$ : Decay rates, typicaly $\beta_1 = 0.9$ 

$\hat{v}$ : second moment (variance) estimate with bias correction, leveraging __RMSprop__, the root of the squared gradient

$$
\hat{m}_t= \frac{m_t}{1-\beta_1^t}
$$
and
$$
m_t= \beta_1 m_{t-t}+(1-\beta_1)\frac{\delta L}{\delta w_t}
$$
$$
\hat{v}_t= \frac{v_t}{1-\beta_2^t}
$$
$$
v_t = \beta_2v_{t-1}+(1-\beta_2)(\frac{\delta L}{\delta w_t})^2
$$

In [6]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
np.random.seed(22)
from tqdm import tqdm

X,y = make_blobs(n_samples=2000, centers= 2, cluster_std= 2)

X_train, y_train = X[:1000],y[:1000]
X_valid, y_valid = X[1000:1500],y[1000:1500]
X_train_valid, y_train_valid = X[:1500], y[:1500]
X_test, y_test = X[1500:], y[1500:]

weights = np.zeros(2)
beta_zero = 0.0
learning_rate = 0.0001
epochs= 300
losses = []



for i in tqdm(range(epochs)):
    z = X_train @ weights + beta_zero

    probability = 1 / (1 + np.exp(- z))

    eps = 1e-12
    loss = np.mean(- (y_train * np.log(probability + eps) + (1 - y_train) * np.log(1 - probability + eps)) )
    losses.append(loss)

    dw = (X_train.T @ (probability - y_train)).mean()
    db = (probability - y_train).mean()

    weights -= learning_rate * dw
    beta_zero -= learning_rate * db

plt.plot(losses)
    


100%|██████████| 300/300 [00:00<00:00, 8444.62it/s]


In [7]:
def sigmoid(z): return 1 / (1 + np.exp(-z))
xx, yy = np.meshgrid(
    np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 200),
    np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 200)
)
grid = np.c_[xx.ravel(), yy.ravel()]
probs = sigmoid(grid.dot(weights) + beta_zero).reshape(xx.shape)

plt.figure()
plt.contourf(xx, yy, probs, 20, cmap="Greens")
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="Greens", edgecolor="black")
plt.title("Logistic Regression — Decision Boundary")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()


# Softmax - multidimensional regression

The  main differnce between logistical regression and softmax regression is that the softmax function takes into account multiple classes and the sum of probabilities that a datapoint belongs to a certain class is 1.
multiclass logistic regression calculates probablity values from z values and then evaluates multiclass cross entropy
$$ z_ik = \beta _0 + x_i \cdot \beta _i$$
$$ p_{i,k} = \frac{e^{-z_{i,k}}}{\sum_{j=1}^{K}e^{z_{i,j}}}$$
cross - Entropy - Loss:
$$ L = \frac{1}{n}\sum_{i=1}^{n}\sum_{j=1}^{K}y_i \cdot \log{p}$$

A matrix of weights is created and fitted to the training data. The shape of the weight matrix is n_parameters x n_classes, in this case 2x3.


In [8]:
from sklearn.datasets import make_blobs

X,y = make_blobs(n_samples=2000, centers= 3, cluster_std= 2)
n_classes = 3
weights = np.zeros((X.shape[1],n_classes))
beta_zero = np.zeros((n_classes))
learning_rate = 0.0001
epochs= 3000
losses = []
y_enc = np.zeros((X.shape[0],n_classes)); y_enc[np.arange(len(y)), y] = 1

eps = 1e-12


def softmax(z):
    z -= np.max(z,axis=1,keepdims = True) #stabalizes calculation by preventing overflow
    p = np.exp(z)/np.sum(np.exp(z),axis=1,keepdims = True)
    return p 

for i in tqdm(range(epochs)):
    z = X @ weights + beta_zero
    p = softmax(z)
    loss = -(np.sum(y_enc * np.log(p+ eps), axis = 1)).mean()
    losses.append(loss)
    dw = X.T @ (p - y_enc)
    db = (p - y_enc).mean() 

    weights -= dw * learning_rate
    beta_zero -= db * learning_rate

plt.plot(losses)


100%|██████████| 3000/3000 [00:01<00:00, 2064.50it/s]


In [9]:
xx, yy = np.meshgrid(
    np.linspace(X[:,0].min()-1, X[:,0].max()+1, 300),
    np.linspace(X[:,1].min()-1, X[:,1].max()+1, 300)
)
grid = np.c_[xx.ravel(), yy.ravel()]
preds = np.argmax(softmax(grid.dot(weights)+beta_zero), axis=1).reshape(xx.shape)

plt.figure()
plt.contourf(xx, yy, preds, n_classes, cmap="Greens")
plt.scatter(X[:,0], X[:,1], c=y, cmap="Greens", edgecolor="black")
plt.title("Softmax Regression — Class Regions")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

In [10]:
print(preds
      )

[[0 0 0 ... 1 1 1]
 [0 0 0 ... 1 1 1]
 [0 0 0 ... 1 1 1]
 ...
 [0 0 0 ... 2 2 2]
 [0 0 0 ... 2 2 2]
 [0 0 0 ... 2 2 2]]


# MLP - multilayer perceptron


A neural network is able to learn more complex patters than a linear model. This is achieved through the usage of nonlinear activation functions.

A multi-layer perceptron is a feed-forwrd neural network that has at least three layers. An input layer, one or more hidden layers and an output layer. 

In [11]:
from sklearn.datasets import make_blobs


X,y = make_blobs(n_samples=2000, centers= 3, cluster_std= 1.2)

X_train, y_train = X[:1000],y[:1000]
X_valid, y_valid = X[1000:1500],y[1000:1500]
X_train_valid, y_train_valid = X[:1500], y[:1500]
X_test, y_test = X[1500:], y[1500:]

xx, yy = np.meshgrid(
    np.linspace(X[:,0].min()-1, X[:,0].max()+1, 300),
    np.linspace(X[:,1].min()-1, X[:,1].max()+1, 300)
)
plt.figure()
plt.scatter(X[:,0], X[:,1], c=y, cmap="Greens", edgecolor="black")
plt.title("Data")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

# MLP xor classification

## nn visualizer

In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt


class LiveMLPPlot:
    def __init__(self, X, y, n_hidden):
        self.X = X
        self.y = y
        self.n_hidden = n_hidden

        self.losses = []

        # -------------------------------------------------
        # Create meshgrid for all decision-boundary plots
        # -------------------------------------------------
        margin = 0.5

        x1_min = X[:, 0].min() - margin
        x1_max = X[:, 0].max() + margin

        x2_min = X[:, 1].min() - margin
        x2_max = X[:, 1].max() + margin

        self.xx, self.yy = np.meshgrid(
            np.linspace(x1_min, x1_max, 150),
            np.linspace(x2_min, x2_max, 150)
        )

        self.grid = np.c_[
            self.xx.ravel(),
            self.yy.ravel()
        ]

        # -------------------------------------------------
        # Layout
        #
        # row 0: hidden-neuron decision boundaries
        # row 1: hidden activations | output activations
        # row 2: loss               | final decision boundary
        # -------------------------------------------------

        n_cols = max(n_hidden, 2)
        plt.ion()
        self.fig, self.axes = plt.subplots(
            3,
            n_cols,
            figsize=(4 * n_cols, 11)
        )

        # Make axes always 2D
        if n_cols == 1:
            self.axes = self.axes[:, np.newaxis]

        # Hidden neuron boundary axes
        self.hidden_boundary_axes = [
            self.axes[0, i] for i in range(n_hidden)
        ]

        # Hide unused axes in first row
        for i in range(n_hidden, n_cols):
            self.axes[0, i].axis("off")

        # -------------------------------------------------
        # Bottom rows:
        # combine several subplot cells
        # -------------------------------------------------

        # Remove original axes
        for ax in self.axes[1:, :].flat:
            ax.remove()

        gs = self.axes[0, 0].get_gridspec()

        self.ax_hidden_activation = self.fig.add_subplot(
            gs[1, :n_cols // 2]
        )

        self.ax_output_activation = self.fig.add_subplot(
            gs[1, n_cols // 2:]
        )

        self.ax_loss = self.fig.add_subplot(
            gs[2, :n_cols // 2]
        )

        self.ax_output_boundary = self.fig.add_subplot(
            gs[2, n_cols // 2:]
        )

        plt.tight_layout()
        plt.show()

    def sigmoid(self,z):
        return 1 / (1 + np.exp(-z))
        
    def update(self, model, epoch, loss):
        """
        model is assumed to contain:

        model.weights_input_hidden
            shape: (2, n_hidden)

        model.bias_hidden
            shape: (1, n_hidden)

        model.weights_hidden_output
            shape: (n_hidden, 1)

        model.bias_output
            shape: (1, 1)

        model.hidden_output
            shape: (n_samples, n_hidden)

        model.predicted_output
            shape: (n_samples, 1)

        model.sigmoid(z)
        """

        self.losses.append(loss)

        # =================================================
        # 1. Hidden-layer activation bar plot
        # =================================================

        ax = self.ax_hidden_activation
        ax.clear()

        # Mean activation of each neuron over training samples
        hidden_activations = np.mean(
            model.hidden_output,
            axis=0
        )

        ax.bar(
            np.arange(self.n_hidden),
            hidden_activations
        )

        ax.set_ylim(0, 1)
        ax.set_xlabel("Hidden neuron")
        ax.set_ylabel("Mean activation")
        ax.set_title("Hidden-layer activations")

        ax.set_xticks(np.arange(self.n_hidden))


        # =================================================
        # 2. Output activation bar plot
        # =================================================

        ax = self.ax_output_activation
        ax.clear()

        output_activation = np.mean(
            model.predicted_output,
            axis=0
        )

        ax.bar(
            np.arange(len(output_activation)),
            output_activation
        )

        ax.set_ylim(0, 1)
        ax.set_xlabel("Output neuron")
        ax.set_ylabel("Mean activation")
        ax.set_title("Output activation")


        # =================================================
        # 3. Loss
        # =================================================

        ax = self.ax_loss
        ax.clear()

        ax.plot(
            np.arange(len(self.losses)),
            self.losses
        )

        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.set_title(f"Training loss — epoch {epoch}")

        ax.grid(alpha=0.3)


        # =================================================
        # 4. Hidden neuron decision boundaries
        # =================================================

        # Linear input to every hidden neuron
        hidden_z = (
            self.grid @ model.weights_input_hidden
            + model.bias_hidden
        )

        # Apply activation
        hidden_grid = self.sigmoid(hidden_z)

        for j, ax in enumerate(self.hidden_boundary_axes):

            ax.clear()

            Z = hidden_grid[:, j].reshape(
                self.xx.shape
            )

            # Continuous activation landscape
            ax.contourf(
                self.xx,
                self.yy,
                Z,
                levels=np.linspace(0, 1, 20),
                alpha=0.6
            )

            # Activation = 0.5 decision boundary
            ax.contour(
                self.xx,
                self.yy,
                Z,
                levels=[0.5],
                linewidths=2
            )

            # Original training points
            ax.scatter(
                self.X[:, 0],
                self.X[:, 1],
                c=self.y.ravel(),
                edgecolors="black"
            )

            ax.set_title(
                f"Hidden neuron {j + 1}"
            )

            ax.set_xlabel("$x_1$")
            ax.set_ylabel("$x_2$")


        # =================================================
        # 5. Final network decision boundary
        # =================================================

        ax = self.ax_output_boundary
        ax.clear()

        # hidden_grid already contains the hidden-layer
        # activation for every point in the meshgrid.

        output_z = (
            hidden_grid @ model.weights_hidden_output
            + model.bias_output
        )

        output_grid = self.sigmoid(output_z)

        Z = output_grid.reshape(self.xx.shape)

        # Probability landscape
        ax.contourf(
            self.xx,
            self.yy,
            Z,
            levels=np.linspace(0, 1, 20),
            alpha=0.6
        )

        # p = 0.5 classification boundary
        ax.contour(
            self.xx,
            self.yy,
            Z,
            levels=[0.5],
            linewidths=3
        )

        ax.scatter(
            self.X[:, 0],
            self.X[:, 1],
            c=self.y.ravel(),
            edgecolors="black"
        )

        ax.set_xlabel("$x_1$")
        ax.set_ylabel("$x_2$")
        ax.set_title("Network decision boundary")


        # =================================================
        # Refresh window
        # =================================================

        self.fig.canvas.draw()
        self.fig.canvas.flush_events()

        plt.pause(0.001)

## neural network

output_error: $\frac{\delta L}{\delta p}$

output_delta: $\frac{\delta L}{\delta p}\cdot \frac{\delta p }{\delta z} = \delta_0$

hidden_error: $\delta_0 \cdot \frac{\delta z_o }{\delta h}  $

hidden_delta: $\delta_0 \cdot \frac{\delta z_o }{\delta h} \cdot \frac{\delta h}{\delta z_i}$



In [4]:
import numpy as np

class NeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size):
        self.input_size= input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        # Filling the initialized weights with random values is somehow super importanz
        self.weights_input_hidden = np.random.randn(self.input_size, self.hidden_size)
        self.weights_hidden_output = np.random.randn(self.hidden_size, self.output_size)

        self.bias_hidden = np.zeros((1, hidden_size))
        self.bias_output = np.zeros((1, output_size))

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def sigmoid_derivative(self, p):
        return p * (1 - p)
    
    def feedforward(self, X):
        self.hidden_activation = np.dot(X, self.weights_input_hidden) + self.bias_hidden
        self.hidden_output = self.sigmoid(self.hidden_activation)

        self.output_activation = np.dot(self.hidden_output, self.weights_hidden_output) + self.bias_output
        self.predicted_output = self.sigmoid(self.output_activation)

        return self.predicted_output

    def backward(self, X, y, learning_rate):
        output_error = self.predicted_output - y
        output_delta = output_error * self.sigmoid_derivative(self.predicted_output)

        hidden_error = np.dot(output_delta, self.weights_hidden_output.T)
        hidden_delta = hidden_error * self.sigmoid_derivative(self.hidden_output)

        self.weights_hidden_output -= np.dot(self.hidden_output.T, output_delta) * learning_rate
        self.bias_output -= np.sum(output_delta, axis=0, keepdims=True) * learning_rate

        self.weights_input_hidden -= np.dot(X.T, hidden_delta) * learning_rate
        self.bias_hidden -= np.sum(hidden_delta, axis=0, keepdims=True) * learning_rate

    def train(self, X, y, epochs, learning_rate):
        for epoch in range(epochs):
            output = self.feedforward(X)

            self.backward(X, y, learning_rate)

            if epoch % 4000 == 0:
                loss = np.mean(np.square(y - output))
                print(f"Epoch {epoch}, Loss: {loss}")

In [15]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([[0], [1], [1], [0]])

nn = NeuralNetwork(2, 4, 1)
nn.train(X, y, epochs=10000, learning_rate=0.1)

output = nn.feedforward(X)
print("Predictions after training:")
print(output)

Epoch 0, Loss: 0.3304233692271607
Epoch 4000, Loss: 0.016235871161372248
Epoch 8000, Loss: 0.0029803580993764418
Predictions after training:
[[0.03927756]
 [0.94773623]
 [0.95932925]
 [0.04657314]]


The exaple above shows a very simple dataset that the neural network trains on. It however shows that a problem, that cannot be solved by a linear solution, is solvable with a neural network. The example below used the same neural network to solve an upscaled version of the ssame problem with an impressive visualization.

__The role of backpropagation__ is to find out, how big the influence of a certain weight is on the correctnes sof the final prediction and in which direction the weight would have to be get moved to improve the result. Therefore a gradient for each weight at each point of the network has to be calculated. 

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

np.random.seed(22)
X, blob_labels = make_blobs(
    n_samples=1000, centers=[[-1, -1], [-1, 1], [1, -1], [1, 1]],
    cluster_std=0.25, random_state=22
)
y = np.array([0, 1, 1, 0])[blob_labels]


model = NeuralNetwork(2, 4, 1)



# backward sums gradients, so normalize the learning rate by sample count.
nn.train(X, y.reshape(-1, 1), epochs=10000, learning_rate=0.1/ len(X))
output = nn.feedforward(X).ravel()
print(f'Training accuracy: {np.mean((output >= 0.5) == y):.2%}')

xx, yy = np.meshgrid(
    np.linspace(X[:,0].min()-1, X[:,0].max()+1, 300),
    np.linspace(X[:,1].min()-1, X[:,1].max()+1, 300)
)
plt.figure()
plt.scatter(X[:,0], X[:,1], c=y, cmap="Greens", edgecolor="black")
plt.title("Data")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

grid = np.c_[xx.ravel(), yy.ravel()]
probs = nn.feedforward(grid).reshape(xx.shape)
hidden_probs = nn.hidden_output.copy()

# Hidden neuron boundaries are their 0.5 activation contours (z = 0).
surfaces = [("XOR", probs)] + [
    (f"h{j + 1}", hidden_probs[:, j].reshape(xx.shape))
    for j in range(nn.hidden_size)
]
for name, activation in surfaces:
    plt.figure()
    plt.contourf(xx, yy, activation, levels=np.linspace(0, 1, 21), cmap="Greens")
    plt.colorbar(label="P(y=1)" if name == "XOR" else f"{name} activation")
    plt.contour(xx, yy, activation, levels=[0.5], colors="black")
    plt.scatter(X[:,0], X[:,1], c=y, cmap="Greens", edgecolor="black")
    plt.title(f"{name} - Decision Boundary")
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.show()


Epoch 0, Loss: 0.03919539505707256
Epoch 4000, Loss: 0.005879357116666591
Epoch 8000, Loss: 0.003431667698101892
Training accuracy: 99.90%


: 

In [6]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

np.random.seed(22)
X, blob_labels = make_blobs(
    n_samples=1000, centers=[[-1, -1], [-1, 1], [1, -1], [1, 1]],
    cluster_std=0.25, random_state=22
)
y = np.array([0, 1, 1, 0])[blob_labels]
class Model_Params_Container:
    def __init__(self, 
                weights_input_hidden, 
                bias_hidden, 
                weights_hidden_output, 
                bias_output, 
                hidden_output, 
                predicted_output):
        self.weights_input_hidden= weights_input_hidden
        self.bias_hidden= bias_hidden
        self.weights_hidden_output= weights_hidden_output
        self.bias_output= bias_output
        self.hidden_output= hidden_output
        self.predicted_output= predicted_output

class NN_Visulaized(NeuralNetwork):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__(input_size, hidden_size, output_size)
        
    def train(self, X,y, epochs, learning_rate):
        visualizer = LiveMLPPlot(
            X,
            y,
            n_hidden= self.weights_input_hidden.shape[1]    
        )
        for epoch in range(epochs):

            output = self.feedforward(X)
            
            loss = np.mean((y - output) ** 2)

            model_params= Model_Params_Container(self.weights_input_hidden, 
                           self.bias_hidden, 
                           self.weights_hidden_output, 
                           self.bias_output, 
                           self.hidden_output, 
                           self.predicted_output)

            if epoch % 100 == 0:
                visualizer.update(
                    model_params,
                    epoch,
                    loss
                )

            self.backward(X, y, learning_rate)
        


model = NN_Visulaized(2, 4, 1)
model.train(X, y.reshape(-1, 1), epochs=10000, learning_rate=0.1)




# NeRF -neural radiance field

# INR - implicit neural representation